In [2]:
import h5py
import pickle
import time
import functools
import argparse

import pandas as pd
import numpy as np

from datetime import datetime
from scipy.sparse import csc_matrix, diags
from k_means_constrained import KMeansConstrained
from scipy.spatial.distance import cdist
from multiprocessing.dummy import Pool as ThreadPool


In [7]:
def loadH5(input_h5, dimred_method="tsne", label_col="CellLine"):
    """
    Load peak matrix and cell embedding from an HDF5 file.

    Parameters
    ----------
    input_h5 : str
        Path to HDF5 file (see README for expected structure).
    dimred_method : str
        Which embedding to load: 'umap' or 'tsne'.
    label_col : str
        Name to give the grouping column (column 3 of the embedding table).
        Defaults to 'CellLine'; set via --label-col to match your dataset.

    Returns
    -------
    input_df : pd.DataFrame
        Per-cell embedding with columns: embedding1, embedding2, CB, <label_col>.
    sparse_peak : scipy.sparse.csc_matrix
        Sparse peak-by-cell accessibility matrix.
    sparse_peak_colnames : np.ndarray of str
        Cell barcodes corresponding to columns of sparse_peak.
    """
    with h5py.File(input_h5, "r") as f:

        dimred_data = f["embedding/%s_df" % dimred_method][:]
        input_df = pd.DataFrame.from_records(dimred_data)
        input_df.columns = ["embedding1", "embedding2", "CB", label_col] #rename here for consistency
        input_df["CB"] = input_df["CB"].astype(str)
        input_df[label_col] = input_df[label_col].astype(str)

        # Load sparse peak matrix (CSC format)
        x = f["peak_matrix/x"][:].astype(np.int16)
        i = f["peak_matrix/i"][:]
        p = f["peak_matrix/p"][:]
        sparse_peak = csc_matrix((x, i, p))
        sparse_peak_colnames = f["peak_matrix/colnames"][:].astype(str)

    return input_df, sparse_peak, sparse_peak_colnames

embedding_df, sparse_peak, sparse_peak_colnames = loadH5(
        "/pollard/data/projects/aseveritt/encode_snatacseq/scBAMpler/test_data/peakmat_input.h5", "umap", label_col="CellLine"
    )


In [9]:
def _cluster_one_cellline(df, cluster_size, seed, cellline):
    """
    Run size-constrained k-means on a single cell group's embedding.

    Clusters that exceed cluster_size after fitting have their outermost
    cells (farthest from centroid) reassigned to cluster -1 (unassigned).

    Parameters
    ----------
    df : pd.DataFrame
        Subset of the full embedding DataFrame for one cell line.
    cluster_size : int
        Target (minimum) cells per cluster.
    seed : int
        Random seed for reproducibility.
    cellline : str
        Cell line label (used only for logging).

    Returns
    -------
    df : pd.DataFrame or None
        DataFrame with a 'Cluster' column added, or None if too few cells.
    """
    
    X = df[["embedding1", "embedding2"]].to_numpy()

    if len(X) < cluster_size:
        return None

    # Constrained k-means clustering
    kmeans = KMeansConstrained(
        n_clusters=len(X) // cluster_size,
        size_min=cluster_size,
        size_max=None,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=seed,
        n_jobs=10,
    ).fit(X)

     # Assign cluster labels
    df["Cluster"] = kmeans.labels_
    cluster_centers = kmeans.cluster_centers_

    # Trim any clusters that still exceed the size limit
    cluster_counts = df["Cluster"].value_counts()
    large_clusters = cluster_counts[cluster_counts > cluster_size].index

    for c in large_clusters:
        mask = df["Cluster"] == c
        sub_df = df.loc[mask, ["embedding1", "embedding2"]]
        point_distances = cdist(
            sub_df.to_numpy(), [cluster_centers[c]], metric="euclidean"
        ).flatten()
        
        num_to_remove = len(point_distances) - cluster_size
        removal_indices = np.argpartition(point_distances, -num_to_remove)[-num_to_remove:]
        idxs_large = [sub_df.index[i] for i in removal_indices]
        df.loc[idxs_large, "Cluster"] = -1

    return df



def constrained_cluster(df, cluster_size=500, seed=42, nproc=8, label_col="CellLine"):
    """
    Cluster cells within each group using size-constrained k-means,
    run in parallel across groups.

    Cells that cannot be assigned to a full cluster are labeled 'm-1'.
    All other clusters receive a globally unique label (m1, m2, ...).

    Parameters
    ----------
    df : pd.DataFrame
        Full embedding DataFrame with columns: embedding1, embedding2, CB, <label_col>.
    cluster_size : int
        Target cells per cluster (default: 500).
    seed : int
        Random seed (default: 42).
    nproc : int
        Number of parallel threads (default: 8).
    label_col : str
        Name of the grouping column (default: 'CellLine').

    Returns
    -------
    df : pd.DataFrame
        Input DataFrame with 'Cluster' column added.
    """
    
    grouped = df.groupby(label_col)

    with ThreadPool(processes=nproc) as pool:
        results = pool.starmap(
            _cluster_one_cellline,
            [(df_sub, cluster_size, seed, cl) for cl, df_sub in grouped],
        )

    df = pd.concat([r for r in results if r is not None], ignore_index=True)
    df["Cluster"] = df["Cluster"].astype(str)

    # Assign globally unique cluster labels
    i = 1
    for (label, sub_df) in df.groupby([label_col, "Cluster"]):
        if label[1] == "-1":
            df.loc[sub_df.index, "Cluster"] = "m-1"
        else:
            df.loc[sub_df.index, "Cluster"] = f"m{i}"
            i += 1

    return df


embedding_df = constrained_cluster(
    embedding_df, cluster_size=500, nproc=4,
    label_col="CellLine"
    )

In [10]:
embedding_df

,embedding1,embedding2,CB,CellLine
0,-17.043638,-9.763289,MCF7#CCCGGAAACTCTGTGA,MCF7
1,-17.026039,-11.008070,MCF7#CGTCGTACTGTCTGGC,MCF7
2,-17.017236,-10.264926,MCF7#AGCCCAGTGAACTAGA,MCF7
3,-16.943134,-9.484824,MCF7#TAGGCCTCTTGAACAA,MCF7
4,-17.438080,-10.063845,MCF7#TATTCAGCTTGACTCA,MCF7
...,...,...,...,...
1106,2.557819,2.403085,HEPG2#TAGACCTGACGTAGAG,HEPG2
1107,5.515608,3.772902,HEPG2#CGACATCCTTTACGCA,HEPG2
1108,-14.367861,-8.924137,HEPG2#TGTGCAATGGCTCAGC,HEPG2
1109,5.672406,-0.051795,HEPG2#ATCTGCCTGTTCGGGC,HEPG2
